In [20]:
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load ByT5 and tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/byt5-small")
byt5 = AutoModelForSeq2SeqLM.from_pretrained("google/byt5-small").to(device)

# Projection layer: SHuBERT (768) → ByT5 d_model
proj = nn.Linear(768, byt5.config.d_model).to(device)


In [21]:
class SignTranslationDataset(Dataset):
    def __init__(self, csv_path, embedding_dir, tokenizer, max_len=64):
        self.df = pd.read_csv(csv_path)
        self.embedding_dir = embedding_dir
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_file = row['video'].replace('.mp4', '.npy')
        text = row['translation']

        emb_path = os.path.join(self.embedding_dir, video_file)
        emb = np.load(emb_path)

        if emb.ndim == 3 and emb.shape[0] == 12:
            emb = np.mean(emb, axis=0)  # SHuBERT stack

        emb = torch.tensor(emb, dtype=torch.float32)
        attention_mask = torch.ones(emb.shape[0], dtype=torch.long)

        label_enc = tokenizer(
            text, padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt"
        )

        return {
            "embedding": emb,
            "attention_mask": attention_mask,
            "labels": label_enc.input_ids.squeeze(0),
        }


In [22]:
csv_path = "sign_translation_dataset.csv"  # Your CSV
embedding_dir = "output_embeddings"

dataset = SignTranslationDataset(csv_path, embedding_dir, tokenizer)
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=lambda x: x)


In [23]:
optimizer = torch.optim.Adam(list(proj.parameters()) + list(byt5.parameters()), lr=2e-5)
byt5.train()
proj.train()

num_epochs = 3

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}")
    for batch in loader:
        embeddings = [proj(item['embedding'].to(device)) for item in batch]
        max_len = max([e.shape[0] for e in embeddings])
        padded = torch.stack([
            torch.cat([e, torch.zeros(max_len - e.shape[0], e.shape[1]).to(device)]) for e in embeddings
        ])
        attn_masks = torch.stack([
            torch.cat([item['attention_mask'], torch.zeros(max_len - item['attention_mask'].shape[0])]).to(device)
            for item in batch
        ]).long()
        labels = torch.stack([item['labels'].to(device) for item in batch])

        outputs = byt5(
            encoder_outputs=(padded,),
            attention_mask=attn_masks,
            labels=labels,
        )

        loss = outputs.loss
        print("Loss:", loss.item())

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.



Epoch 1
Loss: 22.187828063964844
Loss: 24.870840072631836
Loss: 20.511085510253906

Epoch 2
Loss: 23.222301483154297
Loss: 20.84222984313965
Loss: 19.127408981323242

Epoch 3
Loss: 19.548419952392578
Loss: 18.04844093322754
Loss: 18.24947166442871


In [24]:
torch.save(proj.state_dict(), "projection_shubert_to_byt5.pt")
byt5.save_pretrained("byt5_sign_translation")
tokenizer.save_pretrained("byt5_sign_translation")


('byt5_sign_translation/tokenizer_config.json',
 'byt5_sign_translation/special_tokens_map.json',
 'byt5_sign_translation/added_tokens.json')

In [27]:
from transformers.modeling_outputs import BaseModelOutput

def translate_sign(shubert_path):
    emb = np.load(shubert_path)
    if emb.ndim == 3:
        emb = np.mean(emb, axis=0)

    emb = torch.tensor(emb, dtype=torch.float32).to(device)
    emb = proj(emb).unsqueeze(0)  # shape: (1, T, d_model)
    attn_mask = torch.ones(emb.shape[1], dtype=torch.long).unsqueeze(0).to(device)

    encoder_outputs = BaseModelOutput(last_hidden_state=emb)

    output_ids = byt5.generate(
        encoder_outputs=encoder_outputs,
        attention_mask=attn_mask,
        max_length=50,
        num_beams=5,
        early_stopping=True
    )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# Example:
print(translate_sign("output_embeddings/vKddyMCckSE-375.npy"))


[_-_-jj_j-?M_M,___@j____ѣ_У?__
